# CNN + LSTM + CTC: build and check the model

**This is an untrained model check, not an accuracy experiment.** We pass two real training images from each dataset through the model and check CTC loss and gradients. There is no optimizer step, no change to model weights, and no validation/test evaluation.

The CNN finds image features. The bidirectional LSTM reads their left-to-right sequence with context from both directions. A classifier supplies character probabilities. CTC loss compares those probabilities with the correct text; greedy CTC decoding produces text after training.


In [1]:
from pathlib import Path
import sys
import json
import hashlib
import platform
from datetime import datetime, timezone
from importlib.metadata import version

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/load_dataset/iam.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from src.load_dataset.iam import load_iam_dataset
from src.load_dataset.bentham import load_bentham_dataset, REVISION
from src.training.ctc_data import CharacterVocabulary, CTCImagePreparation, CTCLineDataset, collate_ctc_lines
from src.models.cnn_lstm_ctc import (
    CNNLSTMCTCConfig, CNNLSTMCTC, CNNLSTMCTCRecognizer,
    ctc_batch_loss, greedy_ctc_decode,
)

# Exact saved preparation run; do not rebuild vocabularies from held-out text.
PREPARATION = ROOT / "results/cnn_lstm_ctc/data_preparation/20261114T211708884735Z"
prepared_metadata = json.loads((PREPARATION / "metadata.json").read_text(encoding="utf-8"))
SEED = 5112
DEVICE = "cpu"
# A small, fixed training-only check; not an evaluation subset.
SOURCE_INDICES = {"iam": [0, 1], "bentham": [2, 3]}


## Architecture

- Input: height-64 grayscale images; widths may differ.
- CNN: three convolution blocks with 32, 64 and 128 channels. Pooling reduces width by a factor of four. Remaining height is averaged to form a sequence.
- LSTM: two bidirectional layers, hidden size 128 per direction.
- Classifier: character scores, including the CTC blank.
- Output: `[time positions, batch size, character classes]` log probabilities.

Each image is cropped to its actual prepared width before CNN processing. Packed LSTM sequences exclude batch padding. This keeps one image's output independent of how wide its batch companions are.

These are initial architecture choices, not a copied pretrained model or validated best settings. The training vocabularies give IAM 80 output classes and Bentham 94, including blank.


In [2]:
sources = {
    "iam": load_iam_dataset(split="train", revision=prepared_metadata["datasets"]["iam"]["revision"]),
    "bentham": load_bentham_dataset(split="train"),
}
assert REVISION == prepared_metadata["datasets"]["bentham"]["revision"]
checks = {}
models = {}

def weight_digest(model):
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        digest.update(name.encode())
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

for name, source in sources.items():
    saved = prepared_metadata["datasets"][name]
    assert len(source) == saved["train_count"]
    assert source._fingerprint == saved["train_fingerprint"]
    vocabulary = CharacterVocabulary(tuple(saved["vocabulary"]["characters"]))
    preparation = CTCImagePreparation(height=prepared_metadata["mandatory_input"]["height"])
    dataset = CTCLineDataset(source, vocabulary, preparation, indices=SOURCE_INDICES[name])
    batch = collate_ctc_lines([dataset[i] for i in range(len(dataset))])
    config = CNNLSTMCTCConfig(num_classes=vocabulary.num_classes, input_height=preparation.height)
    torch.manual_seed(SEED)
    model = CNNLSTMCTC(config).to(DEVICE)
    before = weight_digest(model)
    scores, input_lengths = model(batch["images"].to(DEVICE), batch["image_widths"])
    assert scores.shape[2] == vocabulary.num_classes
    assert input_lengths.tolist() == model.output_lengths(batch["image_widths"]).tolist()
    torch.testing.assert_close(scores.exp().sum(2), torch.ones_like(scores[:, :, 0]))
    loss = ctc_batch_loss(scores, input_lengths, batch, blank_id=vocabulary.blank_id)
    assert bool(torch.isfinite(loss))
    loss.backward()
    assert all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters())
    assert any(bool(p.grad.abs().sum() > 0) for p in model.cnn.parameters())
    assert any(bool(p.grad.abs().sum() > 0) for p in model.lstm.parameters())
    assert weight_digest(model) == before  # Backward computes gradients; no optimizer changed weights.
    models[name] = model
    checks[name] = {
        "training_source_indices": SOURCE_INDICES[name], "sample_ids": batch["sample_ids"],
        "dataset_revision": saved["revision"], "dataset_fingerprint": source._fingerprint,
        "model": model.to_config(), "vocabulary": vocabulary.to_config(),
        "mandatory_input": preparation.to_config(),
        "parameter_count": sum(p.numel() for p in model.parameters()),
        "image_batch_shape": list(batch["images"].shape), "image_widths": batch["image_widths"].tolist(),
        "log_probability_shape": list(scores.shape), "ctc_input_lengths": input_lengths.tolist(),
        "target_lengths": batch["target_lengths"].tolist(),
        "minimum_ctc_timesteps": batch["minimum_ctc_timesteps"].tolist(),
        "random_initialization_ctc_loss": float(loss.detach()),
        "finite_gradients": True, "weights_unchanged": True, "initial_weight_sha256": before,
    }
    print(f"{name.upper()}: {checks[name]['parameter_count']:,} parameters")
    print("Image batch:", tuple(batch["images"].shape), "-> character scores:", tuple(scores.shape))
    print("CTC time steps:", input_lengths.tolist(), "| required:", batch["minimum_ctc_timesteps"].tolist())
    print("Finite CTC loss and gradients: PASS; weights unchanged: PASS")
    model.zero_grad(set_to_none=True)


IAM: 772,688 parameters
Image batch: (2, 1, 64, 1334) -> character scores: (333, 2, 80)
CTC time steps: [308, 333] | required: [36, 35]
Finite CTC loss and gradients: PASS; weights unchanged: PASS
BENTHAM: 776,286 parameters
Image batch: (2, 1, 64, 787) -> character scores: (196, 2, 94)
CTC time steps: [143, 196] | required: [29, 61]
Finite CTC loss and gradients: PASS; weights unchanged: PASS


## Why repeated letters and blanks matter

This next example uses a **hand-written numerical path**, not a model prediction. It demonstrates decoding only:

`h h _ e e _ l l _ l l _ o o` becomes `hello`.

Adjacent repeated IDs merge first. Blanks are then removed. The blank between the two runs of `l` preserves the genuine double letter.


In [3]:
toy_vocabulary = CharacterVocabulary(tuple("helo"))
path = [1, 1, 0, 2, 2, 0, 3, 3, 0, 3, 3, 0, 4, 4]
# Explicit synthetic log probabilities for a decoder demonstration.
scores = torch.full((len(path), 1, toy_vocabulary.num_classes), -20.0)
for position, character_id in enumerate(path):
    scores[position, 0, character_id] = 0.0
text = greedy_ctc_decode(scores.log_softmax(2), torch.tensor([len(path)]), toy_vocabulary)[0]
assert text == "hello"
print("Illustrative path: h h _ e e _ l l _ l l _ o o")
print("CTC-decoded text:", text)
print("This is a decoder demonstration, not recognition by the untrained model.")


Illustrative path: h h _ e e _ l l _ l l _ o o
CTC-decoded text: hello
This is a decoder demonstration, not recognition by the untrained model.


## Save the check record

We save configuration and check results, not a trained checkpoint. The random-initialization loss above only verifies that computation works; it is not a trained model's CER/WER or a training curve.

**Next:** implement reproducible training/checkpoint selection on the existing training/validation splits. Optional preprocessing comparisons and final test evaluation come after training.


In [4]:
output = ROOT / "results/cnn_lstm_ctc/model_check" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output.mkdir(parents=True, exist_ok=False)
record = {
    "phase": "untrained_architecture_check", "preparation_reference": str(PREPARATION.relative_to(ROOT)),
    "seed": SEED, "device": DEVICE, "optimizer_steps": 0,
    "training_started": False, "validation_loaded": False, "test_loaded": False,
    "python": platform.python_version(),
    "versions": {p: version(p) for p in ("torch", "datasets", "numpy", "Pillow", "jiwer")},
    "checks": checks,
}
(output / "metadata.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print("Saved model-check record:", output)


Saved model-check record: C:\Users\chirayu ubale\OneDrive\Desktop\Research Project\results\cnn_lstm_ctc\model_check\20261114T212646204788Z
